# Anachronism filter (1930s): thin runner for the `scripts/` pipeline

This notebook is a **thin orchestrator**. All the real logic lives in standalone
Python scripts that are stored in the destination dataset repo under `scripts/`.
The notebook uploads them to Hugging Face, clones the repo, and runs each stage as
its own process. This keeps the heavy code (the 460+ term banned list, the fast
matcher, the shard loop) out of the notebook and versioned alongside the data.

- Source:      [`jbduran/think-dataset-clean`](https://huggingface.co/datasets/jbduran/think-dataset-clean)
- Destination: [`jbduran/think-dataset-clean-1930s`](https://huggingface.co/datasets/jbduran/think-dataset-clean-1930s) (created on first run)
- Method: Michael Hla's keyword-filter approach (drop a whole document that mentions
  anything post-1930), seeded from croqaz/vintage-ft-v1 `banned.txt` and adjusted
  from a 1900 to a 1930 cutoff via an allow-list.

## The pipeline scripts (in `scripts/`)

| Script | Stage | What it does |
|---|---|---|
| `config.py` | — | Shared settings (repos, cutoff, scan window). Imported by all; reads env overrides. |
| `common.py` | — | HF auth, `HfApi`, repo/shard helpers. |
| `wipe.py` | 0 | Guarded clean command: delete generated shards/stats/hits/report. |
| `build_list.py` | 1 | Build + count + upload the banned list to `_banned/`. |
| `filter_lib.py` | 2 | Fast matcher (`compile_matchers`, `scan_text`, `should_drop`). |
| `run_filter.py` | 3 | Resumable shard loop: scan, drop whole docs, write shard + stats + hit log. |
| `report.py` | 4 | Aggregate report + README; ranks which terms fired. |

## How state passes between stages

Each script is its own process. They share settings through `config.py` (with env
overrides set in the bootstrap cell) and share the banned list through HF: Stage 1
uploads `_banned/banned_list.txt`; Stages 3 and 4 download it and rebuild the matcher.
That makes every stage independently runnable and naturally resumable.

## Performance note

The matcher deliberately avoids a 400+ term regex alternation (which is
seconds-per-book on multi-MB OCR text). It uses set membership for single words,
first-token gating for phrases, and a capped head+tail scan window — about 35x
faster with identical results. This runs comfortably on a **plain CPU** runtime;
no GPU, no high-RAM needed.


## 1. Install, authenticate, configure

Installs dependencies, logs into Hugging Face (add a **write** token as a Colab
secret named `HF_TOKEN`), exports config as environment variables for the scripts,
and creates the destination repo.


In [ ]:
# === Install dependencies ===================================================
%pip -q install -U datasets huggingface_hub pyarrow tqdm numpy

import os
from pathlib import Path

from huggingface_hub import HfApi, login

try:
    from google.colab import userdata
except Exception:
    userdata = None

# === Authenticate to Hugging Face ==========================================
# Preferred: add a WRITE token as a Colab secret named HF_TOKEN.
HF_TOKEN = None
if userdata is not None:
    try:
        HF_TOKEN = userdata.get("HF_TOKEN")
    except Exception:
        HF_TOKEN = None
HF_TOKEN = HF_TOKEN or os.environ.get("HF_TOKEN")
if not HF_TOKEN:
    from huggingface_hub import notebook_login
    notebook_login()
    HF_TOKEN = HfApi().token

# Export so the stage scripts (separate processes) can authenticate.
os.environ["HF_TOKEN"] = HF_TOKEN
login(token=HF_TOKEN, add_to_git_credential=True)

# === Configuration (exported as env vars for the scripts) ==================
DST_REPO = "jbduran/think-dataset-clean-1930s"
SRC_REPO = "jbduran/think-dataset-clean"
os.environ["SRC_REPO"] = SRC_REPO
os.environ["DST_REPO"] = DST_REPO
os.environ["WORK_DIR"] = "/content/think_1930s_work"

# Scan window + policy (see scripts/config.py for meaning). Tweak here if needed.
os.environ["CUTOFF_YEAR"] = "1930"
os.environ["MIN_BANNED_HITS"] = "1"
os.environ["SCAN_CHARS"] = "300000"
os.environ["SCAN_TAIL_CHARS"] = "50000"

api = HfApi(token=HF_TOKEN)
api.create_repo(repo_id=DST_REPO, repo_type="dataset", exist_ok=True, private=False)
print(f"Authenticated. Destination repo ready: {DST_REPO}")

## 2. Push scripts to HF and clone

Bring the local `scripts/` folder into this Colab session first (drag-and-drop the
folder into the file browser on the left, or mount Google Drive). This cell uploads
it to `scripts/` in the destination repo, then clones the repo so every stage runs
from a clean checkout. Re-running re-syncs the scripts and pulls the latest.


In [ ]:
# === Push the pipeline scripts to Hugging Face, then clone the repo =========
# The pipeline lives in a local `scripts/` folder. Bring it into this Colab
# session first (drag-and-drop the folder into the file browser, or mount Drive).
# This cell uploads it to `scripts/` in the destination repo, then clones the
# repo so every stage runs from a clean checkout on HF.

LOCAL_SCRIPTS = Path("scripts")   # adjust if you placed it elsewhere in Colab

assert LOCAL_SCRIPTS.is_dir(), (
    f"Could not find '{LOCAL_SCRIPTS}/' in this Colab session. Upload the scripts "
    "folder (config.py, common.py, build_list.py, filter_lib.py, run_filter.py, "
    "report.py, wipe.py) into the working directory and re-run this cell."
)

api.upload_folder(
    repo_id=DST_REPO,
    repo_type="dataset",
    folder_path=str(LOCAL_SCRIPTS),
    path_in_repo="scripts",
    commit_message="upload/update 1930s pipeline scripts",
)
print("Uploaded scripts/ to HF.")

# Clone the repo (or pull if already cloned) and run everything from there.
CLONE_DIR = Path("/content/think-dataset-clean-1930s")
if CLONE_DIR.exists():
    !cd "{CLONE_DIR}" && git pull --quiet
else:
    !git clone --quiet "https://huggingface.co/datasets/{DST_REPO}" "{CLONE_DIR}"

RUN_DIR = CLONE_DIR / "scripts"
print(f"Running stages from: {RUN_DIR}")
!ls -la "{RUN_DIR}"

## 3. Stage 0 — clean command (optional, guarded)

Only deletes anything if you edit the cell to set `CONFIRM_WIPE=1`. Use it to
rebuild the dataset from scratch. Leave as-is for a normal resumable run.


In [ ]:
# === Stage 0: clean command (guarded) =======================================
# Deletes generated shards/stats/hits/report from the destination repo so you can
# start from scratch. Does nothing unless you set CONFIRM_WIPE=1 below. Add
# WIPE_BANNED_LIST=1 to also delete the _banned/ list.
#
# IMPORTANT if you previously ran the OLD (flat, over-aggressive) filter: those
# shards were dropped with the wrong rule. Wipe them so the tiered filter can
# reprocess from scratch -- set CONFIRM_WIPE=1 (and WIPE_BANNED_LIST=1 to also
# discard the old flat banned list) for a one-time reset, then set both back to 0.
!cd "{CLONE_DIR}" && CONFIRM_WIPE=0 WIPE_BANNED_LIST=0 python scripts/wipe.py

## 4. Stage 1 — build the banned list

Builds the list (croqaz seed + 1930 allow-list + post-1930 extras), prints the
**total term count**, and uploads it to `_banned/`. Subsequent runs load the cached
list unless you set `FORCE_REBUILD_LIST=1`.


In [ ]:
# === Stage 1: build (or load) the banned list ===============================
# Builds the 1930s banned list from croqaz + allow-list + extras, prints the
# total term count, and uploads it under _banned/ in the destination repo.
# Re-running loads the cached list; set FORCE_REBUILD_LIST=1 to rebuild.
!cd "{CLONE_DIR}" && FORCE_REBUILD_LIST=0 python scripts/build_list.py

## 5. Stage 0.5 — strip footers (dry run, one shard)

Footer/boilerplate removal runs **before** the anachronism filter. This line-level
pass removes reprint/OCR footer lines — URLs, "printed in the United States of
America", "all rights reserved", photocopy / print-on-demand colophons, ISBN lines,
bare page numbers, library stamps — from each document, writing the stripped corpus
to `stripped/` in the destination repo. Whole books are kept; only footer lines go.

**Run this first (one shard).** Then open `strip_samples/shard_00000.jsonl` on HF and
confirm only genuine footer lines were removed — not book text. Docs that would lose
more than 30% of their lines are kept unstripped and flagged (an over-strip guard).


In [ ]:
# === Stage 0.5: strip footers / boilerplate (runs BEFORE the filter) ========
# Reads the clean corpus, removes footer lines (URLs, "printed in USA", "all rights
# reserved", photocopy / print-on-demand colophons, ISBN lines, bare page numbers,
# library stamps) from each document, and writes the stripped corpus to `stripped/`
# in the destination repo. Whole books are kept -- only footer lines are removed.
#
# DRY RUN FIRST: this strips ONE shard (DRY_RUN_LIMIT=1). Then open
# strip_samples/shard_00000.jsonl on HF and confirm only real footer lines were
# removed (not book text). When happy, set DRY_RUN_LIMIT=0 below and re-run to
# strip the rest (already-stripped shards are skipped).
!cd "{CLONE_DIR}" && DRY_RUN_LIMIT=1 python scripts/strip_footers.py

## 6. Stage 0.5 — strip footers (full run)

Once the strip samples look right, strip the rest of the corpus. Already-stripped
shards are skipped, so re-run after any disconnect until the whole corpus is done.
Only then move on to the anachronism filter below, which reads the `stripped/` layer.


In [ ]:
# === Stage 0.5 (full run): strip footers from all remaining shards ==========
# After you've inspected the dry-run strip samples and are happy, run this to
# strip the rest of the corpus. Already-stripped shards are skipped, so it's safe
# to re-run after any Colab disconnect until the whole corpus is stripped. Only
# then move on to the anachronism filter (Stage 3), which reads `stripped/`.
!cd "{CLONE_DIR}" && DRY_RUN_LIMIT=0 python scripts/strip_footers.py

## 7. Stage 3 — anachronism filter dry run (one shard)

Runs the tiered anachronism filter on the **footer-stripped** layer (`stripped/`).
**Run this first** on one shard, then open `hits/shard_00000.jsonl` on HF and check
that the dropped documents are genuine anachronisms, not false positives. If a term
over-fires, adjust its tier in `scripts/build_list.py`, rebuild the list (Stage 1
with `FORCE_REBUILD_LIST=1`), and try again.


In [ ]:
# === Stage 3: main resumable shard loop =====================================
# Reads the FOOTER-STRIPPED layer (stripped/) produced by Stage 0.5 -- so the
# anachronism filter runs on clean text. (SRC_REPO=<dst repo>, SRC_PREFIX=stripped.)
#
# DRY RUN FIRST. This runs ONE shard (DRY_RUN_LIMIT=1), then stop and inspect
# hits/shard_00000.jsonl on HF to confirm the dropped docs are real anachronisms
# (not false positives). The loop skips shards already present in the destination,
# so it resumes automatically after a disconnect.
!cd "{CLONE_DIR}" && SRC_REPO="{DST_REPO}" SRC_PREFIX=stripped DRY_RUN_LIMIT=1 python scripts/run_filter.py

## 8. Stage 3 — anachronism filter full run (all remaining shards)

Once the dry-run hit log looks right, run this to process everything. Completed
shards are skipped, so it resumes automatically after any Colab disconnect — just
re-run this cell until it reports nothing remaining. Each shard prints kept/removed
counts and a running ETA.


In [ ]:
# === Stage 3 (full run): process all remaining shards =======================
# After you've inspected the dry-run hit log and are happy, run this to process
# the rest. DRY_RUN_LIMIT=0 means "all remaining"; completed shards are skipped,
# so this is safe to re-run after any Colab disconnect until everything is done.
# Reads the footer-stripped layer (same as the dry run).
!cd "{CLONE_DIR}" && SRC_REPO="{DST_REPO}" SRC_PREFIX=stripped DRY_RUN_LIMIT=0 python scripts/run_filter.py

## 9. Stage 4 — aggregate report

Summarizes all completed shards, ranks the top firing terms and the top footer
patterns (your audit surfaces), and writes `cleaning_report_1930s.json` + `README.md`
to the destination repo. Safe to run at any point during the run.


In [ ]:
# === Stage 4: aggregate report + README =====================================
# Sums all per-shard stats, ranks which banned terms actually fired, and writes
# cleaning_report_1930s.json + README.md to the destination repo. Safe to run at
# any point -- it reports whatever shards have completed so far.
!cd "{CLONE_DIR}" && python scripts/report.py